Ranking the host galaxy of GW170817 with initial BAYESTAR skymap.

The code was written by Mária Pálfi (marika97@student.elte.hu).

In [1]:
# importing useful packages
import pandas as pd
import numpy as np 
from scipy.stats import norm
import healpy as hp

from ligo.skymap.io import read_sky_map
from astropy import units as u

import astropy_healpix as ah

/media/SSD/anaconda_dir/anaconda3/lib/python3.11/site-packages/ligo/lw/lsctables.py:89: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal


In [2]:
# Read data
hosts = pd.read_csv('../total_host_list_vol_bayestar.csv')

In [3]:
print( 'Number of galaxies inside the 90% localisation volume:',  len( hosts ) )

Number of galaxies inside the 90% localisation volume: 58


In [4]:
# download initial BAYESTAR skymap
url = "https://dcc.ligo.org/public/0146/G1701985/001/bayestar.fits.gz"
skymap = read_sky_map(url, moc=True)

Calculate useful things

In [5]:
uniq = skymap['UNIQ']
prob = skymap['PROBDENSITY']
distnorm = skymap['DISTNORM']
distmu = skymap['DISTMU']
distsigma = skymap['DISTSIGMA']

level, ipix = ah.uniq_to_level_ipix(uniq)
nside = ah.level_to_nside(level)
pixarea = hp.nside2pixarea(nside)

Define function for ranking based on localisation:

In [6]:
def dp_dV_calc(hosts, n, dp_dV, nside, ipix, prob, distmu, distsigma, distnorm, pixarea):
    match_ipix = ah.lonlat_to_healpix( hosts.iloc[n]['_RAJ2000']*u.deg, 
                                      hosts.iloc[n]['_DEJ2000'] *u.deg,
                                      nside, order = 'nested'  )
    i = np.flatnonzero(ipix == match_ipix)[0]
    dp_dV.append( prob[i] * distnorm[i] * 
                     norm( distmu[i], distsigma[i] ).pdf(hosts.iloc[n]['dL']) 
                     / pixarea[i] )

Index of the host galaxy:

In [7]:
hosts.index[hosts.GWGC == 'NGC4993'][0]

2

**Ranking by localization probability**

In [8]:
dp_dV = []
for n in range(len(hosts)): dp_dV_calc(hosts, n, dp_dV, nside, ipix,
                                       prob, distmu, distsigma, distnorm, pixarea)
hosts['dp_dV'] = dp_dV
s_dp_dV = hosts.sort_values( by = ['dp_dV'] , axis = 0, ascending = False ).index

The ranked list of indices:

In [9]:
s_dp_dV

Index([37, 36,  6, 26, 27, 46, 11, 24,  2, 43, 16,  8, 13, 42, 34,  4, 47, 49,
       54, 22,  7, 32, 31,  3, 38, 29, 51, 39,  1,  9, 21,  0, 18, 15, 25, 55,
       40, 12, 44, 23, 10, 19, 14, 53, 30, 28, 35,  5, 57, 33, 17, 48, 41, 56,
       45, 52, 20, 50],
      dtype='int64')

What is the rank of NGC4993 based on localisation alone?

In [10]:
def rank(arr):
    for i in arr:
        if arr[i] == hosts.index[hosts.GWGC == 'NGC4993'][0]:
            print(i+1)

In [11]:
rank( s_dp_dV )

9


## With stellar mass

In [12]:
# read stellar mass data
data = pd.read_csv( '../total_sm.txt', delimiter = '\t', low_memory=False )

In [13]:
# read weights from Artale
prob_M_BBH = pd.read_csv('../Probability_Mstar_BBH_z0p1.dat', delimiter='\t', 
                         header = 0, names = ['logMs', 'P'] )

In [14]:
# Save stellar masses 
Cluver, Jarrett, Kettlety, Cappellari = [], [], [], []
for i in hosts.GLADE_:
    if len( data[data.ID == i]['SMass_Cluver'].values ) == 0:
        Cluver.append( np.nan )
    else:
        Cluver.append( data[data.ID == i]['SMass_Cluver'].values[0] )
    if len( data[data.ID == i]['SMass_Jarrett'].values ) == 0:
        Jarrett.append( np.nan )
    else:
        Jarrett.append( data[data.ID == i]['SMass_Jarrett'].values[0] )
    if len( data[data.ID == i]['SMass_Kettlety'].values ) == 0:
        Kettlety.append( np.nan )
    else:
        Kettlety.append( data[data.ID == i]['SMass_Kettlety'].values[0] )
    if len( data[data.ID == i]['SMass_Cappellari'].values ) == 0:
        Cappellari.append( np.nan )
    else:
        Cappellari.append( data[data.ID == i]['SMass_Cappellari'].values[0] )

In [15]:
# add stellar masses to dataframe, fill nans with zeros
hosts['Cluver'] = Cluver
hosts.Cluver.fillna(0, inplace=True)
hosts['Kettlety'] = Kettlety
hosts.Kettlety.fillna(0, inplace=True)
hosts['Jarrett'] = Jarrett
hosts.Jarrett.fillna(0, inplace=True)
hosts['Cappellari'] = Cappellari
hosts.Cappellari.fillna(0, inplace=True)

Functions to calculate $G_\text{tot}$:

In [16]:
def find_nearest(array, value, return_array):
    array = np.asarray(array)
    idx = (np.abs(array - value)).argmin()
    return return_array[idx]

def pM_Ducoin( hosts, M, dp_dV ):
    pM = M/np.sum(M)
    alpha = np.sum(hosts[M!=0]['dp_dV'])/np.sum(hosts[M!=0]['dp_dV']*pM[M!=0])
    hosts['dp_dV_pM'] = dp_dV/np.sum(dp_dV)*(1+alpha*pM)
    return hosts.sort_values( by = ['dp_dV_pM'] , axis = 0, ascending = False ).index

def pM_Artale( hosts, M, dp_dV ):
    lg_sm = np.log10(M)
    prob_M_vals = []
    for i in range(len(lg_sm)):
        prob_M_vals.append( find_nearest( prob_M_BBH['logMs'], lg_sm[i], prob_M_BBH['P'] ) )
    prob_M_Artale = prob_M_vals/np.sum(prob_M_vals)
    hosts['prob_M_Artale'] = prob_M_Artale
    alpha = np.sum(hosts[~np.isnan(M)]['dp_dV'])/np.sum(hosts[~np.isnan(M)]['dp_dV']*prob_M_Artale[~np.isnan(M)])
    hosts['prob_Artale'] = dp_dV/np.sum(dp_dV) * (1+alpha*hosts.prob_M_Artale)
    return hosts.sort_values( by = ['prob_Artale'] , axis = 0, ascending = False ).index

**Rank of NGC4993 with $M_\ast$ from Jarrett's method**

In [17]:
s_D_Jarrett = pM_Ducoin( hosts, hosts.Jarrett, dp_dV )
s_A_Jarrett = pM_Artale( hosts, Jarrett, dp_dV )
rank( s_D_Jarrett )
rank( s_A_Jarrett )

1
1


**Rank of NGC4993 with $M_\ast$ from Cluver's method**

In [18]:
s_D_Cluver = pM_Ducoin( hosts, hosts.Cluver, dp_dV )
s_A_Cluver = pM_Artale( hosts, Cluver, dp_dV )
rank( s_D_Cluver )
rank( s_A_Cluver )

1
1


**Rank of NGC4993 with $M_\ast$ from Kettlety's method**

In [19]:
s_D_Kettlety = pM_Ducoin( hosts, hosts.Kettlety, dp_dV )
s_A_Kettlety = pM_Artale( hosts, Kettlety, dp_dV )
rank( s_D_Kettlety )
rank( s_A_Kettlety )

1
1


**Rank of NGC4993 with $M_\ast$ from Cappellari's method**

In [20]:
s_D_Cappellari = pM_Ducoin( hosts, hosts.Cappellari, dp_dV )
s_A_Cappellari = pM_Artale( hosts, Cappellari, dp_dV )
rank( s_D_Cappellari )
rank( s_A_Cappellari )

1
1
